## Binary Search Trees + BFS · Study Notes

**Author:** KTH  
**Date:** Aug 20, 2026

---
**Provenance note.** These two topics live in different chapters:

- **Part 1 — BST** is extracted from **Chapter 11: Binary Search Trees**.
- **Part 2 — BFS** is extracted from **Chapter 15: Graphs**, "Graph search"
  section, which is where the book formally treats BFS. Level-order traversal of
  a *binary tree* appears separately as problem 5.2 in Stacks and Queues; the
  tree-level version is included here as a marked supplement.



# Part 1 — Binary Search Trees (Chapter 11)

> The number of trees which can be formed with *n* + 1 given knots
> α, β, γ, … = (*n* + 1)^(*n*−1).
>
> — "A Theorem on Trees," A. Cayley, 1889

## 1. BST fundamentals

BSTs are a **workhorse of data structures** — they can be used to solve almost
every data structures problem reasonably efficiently. They offer the ability to:

- efficiently **search for a key**
- find the **min and max** elements
- look for the **successor or predecessor** of a search key (which itself need
  not be present in the BST)
- **enumerate the keys in a range** in sorted order

**BST vs. sorted array.** BSTs are similar to arrays in that the stored values
(the "keys") are kept in sorted order. However, unlike a sorted array, keys can
be **added to and deleted from a BST efficiently**.

### The BST property

A BST is a binary tree (as defined in Chapter 6) in which the nodes store keys
that are **comparable** — e.g., integers or strings. The keys must respect the
**BST property**:

> The key stored at a node is **greater than or equal to** the keys stored at
> the nodes of its **left subtree**, and **less than or equal to** the keys
> stored in the nodes of its **right subtree**.

Figure 11.1 shows a BST whose keys are the first 16 prime numbers.

### Complexity and balancing

Key lookup, insertion, and deletion take time **proportional to the height** of
the tree, which in the worst case is `O(n)` if insertions and deletions are
naively implemented. However, there are implementations of insert and delete
which **guarantee height `O(log n)`**. These require storing and updating
additional data at the tree nodes. **Red-black trees** are an example of
height-balanced BSTs and are widely used in data structure libraries.

### ⚠️ A common mistake

A common mistake is that an object present in a BST **is updated in place**. The
consequence: a lookup for that object returns false, **even though it's still in
the BST**.

> **Rule:** avoid putting mutable objects in a BST. Otherwise, when a mutable
> object in a BST must be updated, always **remove it from the tree, then update
> it, then add it back**.

*(Compare the identical warning for hash tables in Chapter 9 — same failure
mode, different structure.)*

## 2. The `BSTNode` prototype

In [1]:
class BSTNode:
    def __init__(self, data=None, left=None, right=None):
        self.data, self.left, self.right = data, left, right

In [2]:
# Build the BST from Figure 11.1 — the first 16 prime numbers.
#                        19
#              7                     43
#         3         11         23         47
#       2   5         17          37        -   53
#             13            29    41
#                31
def build(data, left=None, right=None):
    return BSTNode(data, left, right)

n2  = build(2);   n5  = build(5)
n3  = build(3, n2, n5)
n13 = build(13)
n17 = build(17, n13)
n11 = build(11, None, n17)
n7  = build(7, n3, n11)

n29 = build(29);  n41 = build(41)
n31 = build(31)
n29.right = n31                    # 31 lies between 29 and 37 -> right child of 29
n37 = build(37, n29, n41)
n23 = build(23, None, n37)
n53 = build(53)
n47 = build(47, None, n53)
n43 = build(43, n23, n47)

root = build(19, n7, n43)
print("root:", root.data)

root: 19


In [3]:
# Verify the BST property holds: an inorder traversal of a BST yields sorted order.
def inorder(node, out=None):
    out = [] if out is None else out
    if node:
        inorder(node.left, out)
        out.append(node.data)
        inorder(node.right, out)
    return out

keys = inorder(root)
print(keys)
print("sorted:", keys == sorted(keys))
print("count :", len(keys), "keys (the first 16 primes)")

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53]
sorted: True
count : 16 keys (the first 16 primes)


## 3. Binary search trees boot camp

**Searching is the single most fundamental application of BSTs.** Unlike a hash
table, a BST offers the ability to find the **min and max** elements, and find
the **next largest / next smallest** element.

| | BST | Hash table |
|---|---|---|
| lookup, insert, delete | `O(log n)` (library impls) | `O(1)` average |
| min / max | supported | not supported |
| successor / predecessor | supported | not supported |
| iterate in sorted order | `O(n)` | requires sorting |
| space | `O(n)` (slightly more in practice) | `O(n)` |

The following program demonstrates how to check whether a given value is present
in a BST. It's a **nice illustration of the power of recursion** when operating
on BSTs.

In [4]:
def search_bst(tree, key):
    return (tree if not tree or tree.data == key else search_bst(tree.left, key)
            if key < tree.data else search_bst(tree.right, key))

In [5]:
found = search_bst(root, 31)
print("search 31 ->", found.data if found else None)

missing = search_bst(root, 40)
print("search 40 ->", missing)        # None — 40 is not prime, not in the tree

# Trace the search path taken for key 31
def search_path(tree, key):
    path = []
    while tree and tree.data != key:
        path.append(tree.data)
        tree = tree.left if key < tree.data else tree.right
    if tree:
        path.append(tree.data)
    return path

print("path to 31:", search_path(root, 31))

search 31 -> 31
search 40 -> None
path to 31: [19, 43, 23, 37, 29, 31]


**Complexity.** Since the program descends the tree at each step and spends
`O(1)` time per level, the time complexity is **`O(h)`**, where *h* is the height
of the tree.

## 4. Table 11.1 — Top Tips for Binary Search Trees

- With a BST you can **iterate through elements in sorted order in `O(n)` time**
  — regardless of whether it is balanced.
- Some problems need a **combination of a BST and a hash table**. Example: if you
  insert student objects into a BST ordered by GPA, and then a student's GPA
  needs updating and all you have is the student's name and new GPA, you cannot
  find the student by name without a **full traversal**. With an additional hash
  table, you can go directly to the corresponding entry in the tree.
- **The BST property is a *global* property.** A binary tree may have the
  property that each node's key is greater than the key at its left child and
  smaller than the key at its right child — and *still not be a BST*.

In [6]:
# The third tip, made concrete — the classic "local check is not enough" counterexample.
#        20
#      /    \
#    10      30
#           /  \
#         5     40      <- 5 < 30 locally OK, but 5 is in 20's RIGHT subtree!
bad = BSTNode(20, BSTNode(10), BSTNode(30, BSTNode(5), BSTNode(40)))

def locally_ok(node):
    '''Checks only parent-vs-immediate-children. NOT sufficient.'''
    if not node:
        return True
    if node.left and node.left.data > node.data:
        return False
    if node.right and node.right.data < node.data:
        return False
    return locally_ok(node.left) and locally_ok(node.right)

def is_bst(node, low=float('-inf'), high=float('inf')):
    '''Correct: propagates a global range down the tree.'''
    if not node:
        return True
    if not (low <= node.data <= high):
        return False
    return (is_bst(node.left, low, node.data) and
            is_bst(node.right, node.data, high))

print("local check says  :", locally_ok(bad), " <- wrongly accepts")
print("global check says :", is_bst(bad),      " <- correct")
print("inorder of bad tree:", inorder(bad), "-> not sorted, so not a BST")

local check says  : True  <- wrongly accepts
global check says : False  <- correct
inorder of bad tree: [10, 20, 5, 30, 40] -> not sorted, so not a BST


## 5. Know your binary search tree libraries

Some problems in this chapter entail **writing a BST class**; for others you can
use a BST library. **Python does not come with a built-in BST library.**

**`sortedcontainers`** is the best-in-class module for sorted sets and sorted
dictionaries — performant, clean well-documented API, responsive community. Its
underlying data structure is a **sorted list of sorted lists**. Asymptotic time
for inserts and deletes is `O(√n)`, since these entail insertion into a list of
length roughly √n, rather than the `O(log n)` of balanced BSTs. In practice this
is not an issue, since **CPUs are highly optimized for block data movements**.

In the interests of pedagogy the book uses **`bintrees`**, which implements
sorted sets and dictionaries using balanced BSTs. **Any reasonable interviewer
should accept `sortedcontainers` wherever the book uses `bintrees`.**

### `bintrees` functionality

| Method | Meaning |
|---|---|
| `insert(e)` | Inserts new element `e` in the BST. |
| `discard(e)` | Removes `e` from the BST if present. |
| `min_item()` / `max_item()` | Yield the smallest / largest **key-value pair**. |
| `min_key()` / `max_key()` | Yield the smallest / largest **key**. |
| `pop_min()` / `pop_max()` | Remove **and return** the smallest / largest key-value pair. |

It's particularly important to note that these operations take **`O(log n)`**,
since they are backed by the underlying tree.

The book's illustration:

```python
t = bintrees.RBTree([(5, 'Alfa'), (2, 'Bravo'), (7, 'Charlie'),
                     (3, 'Delta'), (6, 'Echo')])
print(t[2])                              # 'Bravo'
print(t.min_item(), t.max_item())        # (2, 'Bravo'), (7, 'Charlie')
t.insert(9, 'Golf')
print(t.min_key(), t.max_key())          # 2, 9
t.discard(3)
a = t.pop_min()                          # a = (2, 'Bravo')
b = t.pop_max()                          # b = (9, 'Golf')
```

> 📝 **Practical note.** `bintrees` is deprecated and unmaintained (its author
> now recommends `sortedcontainers`), and neither ships with Python. The cell
> below reproduces the same sequence with `sortedcontainers` if it's installed,
> and otherwise falls back to a plain `dict` + `sorted` so the notebook still
> runs everywhere.

In [7]:
# Equivalent of the book's bintrees illustration, without the dependency.
try:
    from sortedcontainers import SortedDict
    t = SortedDict([(5, 'Alfa'), (2, 'Bravo'), (7, 'Charlie'),
                    (3, 'Delta'), (6, 'Echo')])
    backend = "sortedcontainers.SortedDict"
    min_item = lambda: t.peekitem(0)
    max_item = lambda: t.peekitem(-1)
except ImportError:
    t = {5: 'Alfa', 2: 'Bravo', 7: 'Charlie', 3: 'Delta', 6: 'Echo'}
    backend = "plain dict fallback"
    min_item = lambda: (min(t), t[min(t)])
    max_item = lambda: (max(t), t[max(t)])

print("backend:", backend)
print("t[2]            :", t[2])
print("min_item/max_item:", min_item(), max_item())

t[9] = 'Golf'                                   # insert(9, 'Golf')
print("after insert 9   :", dict(sorted(t.items())))
print("min_key/max_key  :", min(t), max(t))

t.pop(3, None)                                  # discard(3)
print("after discard 3  :", dict(sorted(t.items())))

a = min_item(); t.pop(a[0])                     # pop_min()
b = max_item(); t.pop(b[0])                     # pop_max()
print("popped:", a, "and", b)
print("final            :", dict(sorted(t.items())))

backend: sortedcontainers.SortedDict
t[2]            : Bravo
min_item/max_item: (2, 'Bravo') (7, 'Charlie')
after insert 9   : {2: 'Bravo', 3: 'Delta', 5: 'Alfa', 6: 'Echo', 7: 'Charlie', 9: 'Golf'}
min_key/max_key  : 2 9
after discard 3  : {2: 'Bravo', 5: 'Alfa', 6: 'Echo', 7: 'Charlie', 9: 'Golf'}
popped: (2, 'Bravo') and (9, 'Golf')
final            : {5: 'Alfa', 6: 'Echo', 7: 'Charlie'}


# Part 2 — BFS (Chapter 15: Graphs, "Graph search")

> Concerning these bridges, it was asked whether anyone could arrange a route in
> such a way that he would cross each bridge once and only once.
>
> — "The solution of a problem relating to the geometry of position," L. Euler, 1741

## 6. Graph search

Computing **vertices reachable from other vertices** is a fundamental operation,
performed in one of two idiomatic ways: **depth-first search (DFS)** and
**breadth-first search (BFS)**.

**Both have linear time complexity — `O(|V| + |E|)`.**

**Space complexity of DFS is `O(|V|)`.** In the worst case there is a path from
the initial vertex covering all vertices without repeats, and the DFS edges
selected correspond to this path. (This space is *implicitly allocated on the
function call stack*.)

**Space complexity of BFS is also `O(|V|)`**, since in the worst case there is an
edge from the initial vertex to all remaining vertices, implying they will all be
in the **BFS queue simultaneously** at some point.

### What distinguishes them

DFS and BFS differ in the **additional information they provide**:

| | DFS | BFS |
|---|---|---|
| Backing structure | Call stack / explicit stack | **Queue** |
| Time | `O(\|V\| + \|E\|)` | `O(\|V\| + \|E\|)` |
| Space | `O(\|V\|)` | `O(\|V\|)` |
| Provides | Cycle detection; **discovery time** and **finishing time** for vertices | **Distances from the start vertex** |

### When to reach for which (Table 15.1)

- Some graph problems entail **analyzing structure** — e.g., looking for cycles
  or connected components. **DFS works particularly well** for these.
- Some graph problems are related to **optimization** — e.g., finding the
  shortest path from one vertex to another. **BFS**, Dijkstra's shortest path
  algorithm, and minimum spanning tree are appropriate for optimization problems.
- More generally, consider using a graph whenever you need to analyze any
  **binary relationship between objects** — interlinked webpages, followers in a
  social graph, etc. Quite often the problem reduces to a well-known graph
  problem.

## 7. BFS on a graph

The defining property: BFS visits vertices in **nondecreasing order of distance**
(in edges) from the start. That's precisely why it computes shortest paths in
unweighted graphs and DFS does not.

In [8]:
import collections

# An undirected graph as an adjacency list
graph = {
    'a': ['b', 'c'],
    'b': ['a', 'd'],
    'c': ['a', 'e'],
    'd': ['b', 'h'],
    'e': ['c', 'd'],
    'h': ['d'],
}

def bfs_distances(graph, start):
    '''Return {vertex: distance in edges from start} for all reachable vertices.'''
    dist = {start: 0}
    q = collections.deque([start])
    while q:
        u = q.popleft()                  # FIFO is what makes this breadth-first
        for v in graph.get(u, []):
            if v not in dist:            # first time seen == shortest distance
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

print(bfs_distances(graph, 'a'))

{'a': 0, 'b': 1, 'c': 1, 'd': 2, 'e': 2, 'h': 3}


In [9]:
def bfs_shortest_path(graph, start, goal):
    '''Reconstruct one shortest path by recording each vertex's predecessor.'''
    if start == goal:
        return [start]
    parent = {start: None}
    q = collections.deque([start])
    while q:
        u = q.popleft()
        for v in graph.get(u, []):
            if v not in parent:
                parent[v] = u
                if v == goal:            # BFS: first arrival is a shortest one
                    path = [v]
                    while parent[path[-1]] is not None:
                        path.append(parent[path[-1]])
                    return path[::-1]
                q.append(v)
    return None

print("shortest a -> h:", bfs_shortest_path(graph, 'a', 'h'))

shortest a -> h: ['a', 'b', 'd', 'h']


### The contrast, run side by side

DFS reaches every vertex just as BFS does — but the *order* it reaches them in
carries no distance information, so the path it finds may be far from shortest.

In [10]:
def dfs_order(graph, start, seen=None, out=None):
    seen = set() if seen is None else seen
    out = [] if out is None else out
    if start in seen:
        return out
    seen.add(start)
    out.append(start)
    for v in graph.get(start, []):
        dfs_order(graph, v, seen, out)
    return out

def bfs_order(graph, start):
    seen, out, q = {start}, [], collections.deque([start])
    while q:
        u = q.popleft()
        out.append(u)
        for v in graph.get(u, []):
            if v not in seen:
                seen.add(v)
                q.append(v)
    return out

print("DFS visit order:", dfs_order(graph, 'a'))
print("BFS visit order:", bfs_order(graph, 'a'), " <- grouped by distance from 'a'")

DFS visit order: ['a', 'b', 'd', 'h', 'c', 'e']
BFS visit order: ['a', 'b', 'c', 'd', 'e', 'h']  <- grouped by distance from 'a'


## 8. BFS on a binary tree

Level-order traversal of a binary tree is the same algorithm with the adjacency
list replaced by `left`/`right` children. Snapshotting `len(q)` before expanding
is what separates one level from the next.

For a tree, the space bound sharpens usefully: BFS holds at most one level at a
time, so its space is **`O(w)`** where *w* is the maximum **width**. Note this is
*not* comparable to DFS's `O(h)` in general — for a balanced tree the last level
holds ~n/2 nodes (BFS `O(n)`, DFS `O(log n)`), while for a skewed tree the
reverse holds (BFS `O(1)`, DFS `O(n)`).

In [11]:
def bfs_levels(node):
    '''Return a list of levels, each a list of keys at that depth.'''
    if not node:
        return []
    result, q = [], collections.deque([node])
    while q:
        level = []
        for _ in range(len(q)):          # snapshot this level's size first
            n = q.popleft()
            level.append(n.data)
            if n.left:
                q.append(n.left)
            if n.right:
                q.append(n.right)
        result.append(level)
    return result

# Run on the Figure 11.1 BST from Part 1
for depth, level in enumerate(bfs_levels(root)):
    print(f"depth {depth}: {level}")

depth 0: [19]
depth 1: [7, 43]
depth 2: [3, 11, 23, 47]
depth 3: [2, 5, 17, 37, 53]
depth 4: [13, 29, 41]
depth 5: [31]
